<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/%EC%B5%9C%EC%A2%85%EC%8B%9C%EC%8A%A4%ED%85%9C_%ED%95%99%EA%B5%90%EC%95%88%EC%A0%84%EC%82%AC%EA%B3%A0%EC%98%88%EB%B0%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 사고형태별 예방대책 CSV 만들기

import os
import time
import pandas as pd
from tqdm.auto import tqdm
from google import genai

DATA_FILE = "/content/통합_사고예방_데이터.csv"
TEMPLATE_FILE = "/content/사고형태별_예방대책.csv"
GEMINI_MODEL = "gemini-3.6-flash"

# --------------------------------------------------
# 1. 데이터 불러오기
# --------------------------------------------------
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")

if "사고형태" not in df.columns:
    raise KeyError(
        f"'사고형태' 컬럼이 없습니다.\n현재 컬럼:\n{df.columns.tolist()}"
    )

accident_types = sorted(
    df["사고형태"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("사고형태 개수:", len(accident_types))
print(accident_types)

사고형태 개수: 29
['1미터 미만의 높이에서 떨어짐', '1미터 이상의 높이에서 떨어짐', '감전', '고온의 물체·물질 접촉·흡입·섭취', '고정된 물체와의 부딪힘', '곤충·식물 등에 쏘임', '교통사고', '그밖의 손상 사고', '긁힘, 찔림', '기타 호흡 곤란', '넘어짐', '동물에게 물림(사람 포함)', '물건을 운반하는 중 충격을 가함', '물체 사이에 끼임·눌림', '베임, 절단', '사람 사이에 끼임·눌림', '사람과의 부딪힘', '스포츠 활동 중 충격을 가함', '식중독', '움직이는 물체와의 부딪힘', '이동 중 충격을 가함', '이물질 섭취로 인한 질병', '이물질 접촉에 의한 피부염', '이물질에 의한 질식', '익사·익수', '일사병, 열사병', '저온의 물체(드라이아이스 등)·물질 접촉', '추위에 장시간 노출', '화학물질 접촉·흡입·섭취']


In [6]:
# API 키 연결

# Colab 보안 비밀에서 가져오는 경우
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None

# 환경변수에 저장한 경우도 지원
if not api_key:
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY가 없습니다. "
        "Colab 왼쪽 열쇠 아이콘에서 GEMINI_API_KEY를 등록하세요."
    )

client = genai.Client(api_key=api_key)

print("✅ Gemini 연결 완료")

✅ Gemini 연결 완료


In [7]:
# 29개 생성
# 2. 기존 생성 결과가 있으면 이어서 사용
# --------------------------------------------------
if os.path.exists(TEMPLATE_FILE):
    old_template_df = pd.read_csv(
        TEMPLATE_FILE,
        encoding="utf-8-sig"
    )

    template_cache = dict(
        zip(
            old_template_df["사고형태"].astype(str).str.strip(),
            old_template_df["기본예방대책"].astype(str)
        )
    )

    print("기존 생성 결과:", len(template_cache), "개")

else:
    template_cache = {}


# --------------------------------------------------
# 3. 사고형태별 기본 예방대책 생성
# --------------------------------------------------
for accident_type in tqdm(
    accident_types,
    desc="사고형태별 예방대책 생성"
):
    # 이미 생성한 사고형태는 건너뜀
    if (
        accident_type in template_cache
        and str(template_cache[accident_type]).strip()
    ):
        continue

    prompt = f"""
당신은 학교안전사고 예방관리 전문가입니다.

다음 사고형태에 대해 학교 현장에서 실제로 적용할 수 있는
구체적이고 일반화 가능한 예방대책을 작성하세요.

사고형태: {accident_type}

학교급과 사고장소에 따른 세부 조치는 다른 규칙에서 추가되므로,
여기서는 사고형태 자체에 공통으로 적용할 수 있는 핵심 대책을 작성하세요.

아래 형식을 반드시 지키세요.

### 시설·환경 관리
- 구체적인 실행대책 2개

### 학생 행동·안전교육
- 구체적인 실행대책 2개

### 감독·운영 관리
- 구체적인 실행대책 2개

### 사고 발생 시 초기 대응
- 구체적인 대응조치 2개

작성 조건:
- 학교에서 실제 실행 가능한 행동 중심
- '주의한다', '조심한다' 같은 모호한 표현만 사용하지 말 것
- 각 대책은 담당자가 무엇을 해야 하는지 구체적으로 작성
- 확인되지 않은 통계, 법령명, 기관명은 만들지 말 것
- 전체 분량은 500~700자
"""

    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )

        plan = response.text.strip()

        if not plan:
            raise ValueError("빈 응답이 반환되었습니다.")

        template_cache[accident_type] = plan

        # 하나 생성할 때마다 저장
        save_df = pd.DataFrame({
            "사고형태": list(template_cache.keys()),
            "기본예방대책": list(template_cache.values())
        })

        save_df.to_csv(
            TEMPLATE_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        time.sleep(1)

    except Exception as error:
        print(f"\n❌ 생성 실패: {accident_type}")
        print(error)
        time.sleep(5)


print("\n✅ 생성 완료")
print("저장된 예방대책:", len(template_cache), "개")
print("파일 위치:", TEMPLATE_FILE)

사고형태별 예방대책 생성:   0%|          | 0/29 [00:00<?, ?it/s]


❌ 생성 실패: 물체 사이에 끼임·눌림
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 53.808355224s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.

In [9]:
# 결과 확인

template_df = pd.read_csv(
    "/content/사고형태별_예방대책.csv",
    encoding="utf-8-sig"
)

print(template_df.shape)
display(template_df.head())



(13, 2)


,사고형태,기본예방대책
0,1미터 미만의 높이에서 떨어짐,"### 시설·환경 관리\n- 계단, 단상, 낮은 교구 주변 바닥에 충격 흡수 매트를..."
1,1미터 이상의 높이에서 떨어짐,"### 시설·환경 관리\n- 낙상 위험이 있는 1m 이상 높이의 난간, 계단, 단상..."
2,감전,"### 시설·환경 관리\n- 모든 교실 및 특별실 콘센트에 안전 커버를 설치하고, ..."
3,고온의 물체·물질 접촉·흡입·섭취,### 시설·환경 관리\n- 가열 기기 및 온수 배관 주변에 비접촉식 안전 보호망을...
4,고정된 물체와의 부딪힘,"### 시설·환경 관리\n- 주요 통로 및 체육시설 내 기둥, 모서리 등 충돌 위험..."


In [46]:
# 기존 streamlit 삭제!pkill -f streamlit || true

^C


In [ ]:
# 1. Streamlit 설치
!pip install -q streamlit

# 2. Cloudflared 설치
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared


In [47]:
# Streamlit 실행

!streamlit run /content/app.py \
  --server.port 8501 \
  --server.headless true \
  > /content/logs.txt 2>&1 &

In [ ]:
!cloudflared tunnel --url http://localhost:8501


2026-07-22T16:59:08Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-22T16:59:08Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-22T16:59:14Z INF +--------------------------------------------------------------------------------------------+
2026-07-22T16:59:14Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-22T16:59:14Z INF |  https://collection-boulevard-yet-screenshot.trycloudf